# Config

In [1]:
!pip install nltk==3.9.1
!pip install mlflow==3.3.1

import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


# 3) Split dataset

In [2]:
from collections import Counter

def to_serializable(obj):
    if hasattr(obj, "tolist"):
        return obj.tolist()
    return obj

def count_and_eval(labels): 
    c = Counter(labels)
    p = c[0]/c[1]
    print(c, p)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
import json
from collections import Counter

path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

df = df[df["Interdisciplinario"] != "INDEFINIDO"]
le = LabelEncoder()

df["labels"] = le.fit_transform(df["Interdisciplinario"])

# Crear la clase combinada como categoría
df["stratify"] = df[["labels", "Español"]].astype(str).agg("_".join, axis=1)
# Codificar en números
df["stratify"] = df["stratify"].astype("category").cat.codes

# Convertir a arrays
ids = df["Código VRID"].to_numpy()
strat_classes = df["stratify"].to_numpy()

# Train/Test split (ids y labels en paralelo)
idx_train, idx_test, y_train, y_test = train_test_split(
    ids,
    strat_classes,
    test_size=0.2,
    random_state=7,
    stratify=strat_classes
)


print(ids.shape)
print(idx_train.shape)
print(idx_train.shape[0]+idx_test.shape[0])

print("Train:", Counter(y_train))
print("Test:", Counter(y_test))
"""
print("Train:", Counter(idx_train))
print("Test:", Counter(idx_test))

count_and_eval(idx_train)
count_and_eval(idx_test)
"""

# Crear folds sobre train
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

folds = []
for fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):
    train_ids = idx_train[train_pos]   # array de IDs
    val_ids = idx_train[val_pos]       # array de IDs
    folds.append(val_ids)

print("Test size:", len(idx_test))
print("Fold 0 - Val size:", len(folds[0]))

#Guardar index en diccionario
dataset_index = {
    "Train": idx_train,
    "Test": idx_test,
    "kfolds": folds 
}
filepath=os.path.join(path, "langs_3folds.json")

# Guardar
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)


(964,)
(771,)
964
Train: Counter({3: 295, 0: 169, 1: 158, 2: 149})
Test: Counter({3: 74, 0: 43, 1: 39, 2: 37})
Test size: 193
Fold 0 - Val size: 257


# 4) TF-ID feature extractor 

## Train

In [34]:
import numpy as np
import pandas as pd
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset

def gen_dataset(codes_vrid, df):
    #Selección unicamente de elementos de df que se encuentren en codes_vrid
    df = df[df["Código VRID"].isin(codes_vrid)].copy()

    #Creación de index en función de orden de los datos
    df['idx'] = np.arange(0, df.shape[0])

    #Generación de datasets
    X = df["text_for_embedding_translated"].to_list()
    y = df["Interdisciplinario"].to_list()
    return X, y, df

def decoder_vrid(fold_codes, df_decode):
    """
    Decodifica fold_codes usando df_decode.
    fold_codes : lista o array con índices (ej. [0, 2, 5])
    df_decode  : DataFrame con columnas ['idx', 'Código VRID']

    Devuelve un numpy.array con los códigos VRID correspondientes.
    """
    # Crear un diccionario {Código VRID: código}
    mapping = df_decode.set_index("Código VRID")["idx"].to_dict()

    # Mapear los fold_codes a códigos (ignora los que no existan en mapping)
    decoded = [mapping[c] for c in fold_codes if c in mapping]

    return np.array(decoded)

class CvCustom():
    def __init__(self, df_decode, n_splits = None):
        #Dict codes
        self.df_decode=df_decode
        #Lectura de index de separacion de conjuntos train/test
        path = "/tmp/data"
        filepath=os.path.join(path, "langs_3folds.json")
        with open(filepath, "r", encoding="utf-8") as f:
            dataset_index = json.load(f)
        folds_codes = dataset_index["kfolds"]
        self.n_splits=len(folds_codes)
        #Define index for kfolds
        self.kfolds = []
        for i in range(self.n_splits):
            self.kfolds.append(decoder_vrid(folds_codes[i], self.df_decode))
            
        #Save al idx
        self.all_idx = np.array([i for fold in self.kfolds for i in fold])
        print(self.all_idx.shape)
    
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        for i in range(self.n_splits):
            test_idx = self.kfolds[i]
            train_idx = np.setdiff1d(self.all_idx, test_idx) 
            yield train_idx, test_idx


In [35]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "langs_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)
df = df[df["Español"]==False]

In [36]:
from sklearn.preprocessing import LabelEncoder
from models.TIFD import gen_TFID_vectors
from utils.dataset import gen_dataset
import numpy as np

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

(318, 10591) (80, 10591)


In [37]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({0: 169, 1: 149})
📊 test: Counter({0: 43, 1: 37})
(318,)
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'l

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.58, 'std_test_score': 0.05}
RandomForestClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.58, 'std_test_score': 0.05}
SVC: {'mean_test_score': 0.59, 'std_test_score': 0.04}


In [38]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6625, 'precision': 0.6041666666666666, 'recall': 0.7837837837837838, 'f1_macro': 0.6611764705882353, 'cm': array([[24, 19],
       [ 8, 29]]), 'f1_es': 0.0, 'f1_en': 0.6595882352941176, 'cm_es': array([], shape=(0, 0), dtype=int64), 'cm_en': array([[24, 19],
       [ 8, 29]])}
RandomForestClassifier
{'accuracy': 0.725, 'precision': 0.6530612244897959, 'recall': 0.8648648648648649, 'f1_macro': 0.7234443746071653, 'cm': array([[26, 17],
       [ 5, 32]]), 'f1_es': 0.0, 'f1_en': 0.7218887492143307, 'cm_es': array([], shape=(0, 0), dtype=int64), 'cm_en': array([[26, 17],
       [ 5, 32]])}
XGBClassifier
{'accuracy': 0.6, 'precision': 0.5714285714285714, 'recall': 0.5405405405405406, 'f1_macro': 0.595959595959596, 'cm': array([[28, 15],
       [17, 20]]), 'f1_es': 0.0, 'f1_en': 0.598989898989899, 'cm_es': array([], shape=(0, 0), dtype=int64), 'cm_en': array([[28, 15],
       [17, 20]])}
SVC
{'accuracy': 0.675, 'precision': 0.6222222222222222, 'recall': 0.75

## Save

In [39]:
exp_info = {
    'exp_name': "Bayesiansearchcv_TFID_lang",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "data_lang": "ENG"
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/03 16:32:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/5/runs/22ddd73b83df42ffb541efbc4ff6680e
🧪 View experiment at: http://mlflow-server:5000/#/experiments/5
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/03 16:32:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/5/runs/27c9e1a5a5ab425cb31419bdc362460c
🧪 View experiment at: http://mlflow-server:5000/#/experiments/5
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/03 16:32:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/5/runs/3fd5289ce0e349a0ae24e615790b9ea2
🧪 View experiment at: http://mlflow-server:5000/#/experiments/5
📝 Registrando modelo en MLflow: SVC


2025/09/03 16:32:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/5/runs/5e72fbb18fcc4de1b7d905c08b25a2f1
🧪 View experiment at: http://mlflow-server:5000/#/experiments/5


# 4) SPECTER 

In [53]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "langs_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)
df = df[df["Español"]==False]

In [54]:
#del gen_dataset
from utils.dataset import gen_dataset
from models.specter import embed_texts
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# 2) Calcular embeddings
# Parámetros modelo
BASE_MODEL = "allenai/specter2_base"
#ADAPTER_NAME = "allenai/specter2"
ADAPTER_NAME="allenai/specter2_classification"
X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)

print(X_train.shape, X_test.shape)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(318, 768) (80, 768)


In [55]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({0: 169, 1: 149})
📊 test: Counter({0: 43, 1: 37})
(318,)
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', '

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.6, 'std_test_score': 0.01}
RandomForestClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.64, 'std_test_score': 0.03}
SVC: {'mean_test_score': 0.61, 'std_test_score': 0.02}


In [56]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.5875, 'precision': 0.54, 'recall': 0.7297297297297297, 'f1_macro': 0.5843174303259329, 'cm': array([[20, 23],
       [10, 27]]), 'f1_es': 0.0, 'f1_en': 0.5815895134624468, 'cm_es': array([], shape=(0, 0), dtype=int64), 'cm_en': array([[20, 23],
       [10, 27]])}
RandomForestClassifier
{'accuracy': 0.6, 'precision': 0.5555555555555556, 'recall': 0.6756756756756757, 'f1_macro': 0.5997498436522828, 'cm': array([[23, 20],
       [12, 25]]), 'f1_es': 0.0, 'f1_en': 0.5989993746091308, 'cm_es': array([], shape=(0, 0), dtype=int64), 'cm_en': array([[23, 20],
       [12, 25]])}
XGBClassifier
{'accuracy': 0.6375, 'precision': 0.5952380952380952, 'recall': 0.6756756756756757, 'f1_macro': 0.6374433505235193, 'cm': array([[26, 17],
       [12, 25]]), 'f1_es': 0.0, 'f1_en': 0.6377832473824034, 'cm_es': array([], shape=(0, 0), dtype=int64), 'cm_en': array([[26, 17],
       [12, 25]])}
SVC
{'accuracy': 0.575, 'precision': 0.5272727272727272, 'recall': 0.7837837837837

## Save

In [57]:
exp_info = {
    'exp_name': "SPECTER_TFID_lang",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "data_lang": "ENG"
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/03 17:00:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/6/runs/03a1ea5885df40d1b85178b5bd5b2185
🧪 View experiment at: http://mlflow-server:5000/#/experiments/6
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/03 17:00:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/6/runs/7a1c0dedee93488885601a72cb3ed8c3
🧪 View experiment at: http://mlflow-server:5000/#/experiments/6
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/03 17:00:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/6/runs/2483c27ed36a4f34acba4bd446956d79
🧪 View experiment at: http://mlflow-server:5000/#/experiments/6
📝 Registrando modelo en MLflow: SVC


2025/09/03 17:00:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/6/runs/822fbb19b128499ca10532948786d970
🧪 View experiment at: http://mlflow-server:5000/#/experiments/6
